# Failure-Aware Design in LangGraph for Telecom Domain
## Retry, Fallback, Circuit Breaker, Model vs Tool Failure

This notebook is a step-by-step hands-on lab for building a failure-aware telecom agent using:

- LangGraph
- tools
- MCP-style tool descriptors
- telecom APIs with simulated failures

## Objective

Build a telecom assistant that can handle a query like:

> "My mobile internet is slow in Pune. What should support do next?"

and continue to operate safely when:
- an API fails temporarily
- a tool keeps failing
- the model generation step fails
- a fallback path is needed

## What this notebook covers

1. Tool schema and contracts
2. MCP basics
3. Retry pattern
4. Fallback pattern
5. Circuit breaker pattern
6. Difference between model failure and tool failure
7. API failure simulation
8. Failure-aware LangGraph workflow

In [ ]:
# Uncomment if needed
# %pip install -U langgraph pydantic

## Step 1 - Imports

In [18]:
from __future__ import annotations

from typing import TypedDict, Dict, Any, Optional, Literal, List
from dataclasses import dataclass
from pprint import pprint

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

## Step 2 - Concepts

### Retry
Try the same tool again for a limited number of times.

### Fallback
Use a backup tool or safe default when the primary tool fails.

### Circuit breaker
After repeated failures, stop calling the broken dependency and route directly to fallback.

### Tool failure vs model failure
Tool failure means an external dependency fails.
Model failure means the generation step itself fails.

## Step 3 - Tool schemas and contracts

In [19]:
class CustomerProfileInput(BaseModel):
    customer_id: str

class CustomerProfileOutput(BaseModel):
    customer_id: str
    name: str
    city: str
    plan: str
    network_type: str
    recent_complaints: int

class OutageCheckInput(BaseModel):
    city: str
    issue_type: str

class OutageCheckOutput(BaseModel):
    city: str
    issue_type: str
    outage_detected: bool
    planned_maintenance: bool
    summary: str

class PolicySearchInput(BaseModel):
    query_text: str
    city_filter: Optional[str] = None
    issue_filter: Optional[str] = None

class PolicySearchOutput(BaseModel):
    top_match: str
    source: str

class TicketCreateInput(BaseModel):
    customer_id: str
    city: str
    issue_type: str
    diagnosis: str
    priority: Literal["P1", "P2", "P3"]

class TicketCreateOutput(BaseModel):
    ticket_id: str
    status: str
    queue: str

## Step 4 - MCP-style tool catalog

In [20]:
tool_catalog = [
    {
        "name": "get_customer_profile",
        "description": "Fetch telecom customer profile from CRM",
        "input_schema": CustomerProfileInput.model_json_schema(),
        "output_schema": CustomerProfileOutput.model_json_schema(),
    },
    {
        "name": "check_outage_primary",
        "description": "Primary outage API for telecom incident lookup",
        "input_schema": OutageCheckInput.model_json_schema(),
        "output_schema": OutageCheckOutput.model_json_schema(),
    },
    {
        "name": "search_policy",
        "description": "Telecom SOP and RCA retrieval tool",
        "input_schema": PolicySearchInput.model_json_schema(),
        "output_schema": PolicySearchOutput.model_json_schema(),
    },
    {
        "name": "create_ticket",
        "description": "Create telecom support ticket",
        "input_schema": TicketCreateInput.model_json_schema(),
        "output_schema": TicketCreateOutput.model_json_schema(),
    },
]

pprint(tool_catalog[0])

{'description': 'Fetch telecom customer profile from CRM',
 'input_schema': {'properties': {'customer_id': {'title': 'Customer Id',
                                                 'type': 'string'}},
                  'required': ['customer_id'],
                  'title': 'CustomerProfileInput',
                  'type': 'object'},
 'name': 'get_customer_profile',
 'output_schema': {'properties': {'city': {'title': 'City', 'type': 'string'},
                                  'customer_id': {'title': 'Customer Id',
                                                  'type': 'string'},
                                  'name': {'title': 'Name', 'type': 'string'},
                                  'network_type': {'title': 'Network Type',
                                                   'type': 'string'},
                                  'plan': {'title': 'Plan', 'type': 'string'},
                                  'recent_complaints': {'title': 'Recent '
                              

## Step 5 - Mock telecom data

In [21]:
CUSTOMERS = {
    "CUST_1001": {
        "customer_id": "CUST_1001",
        "name": "Ravi Patil",
        "city": "Pune",
        "plan": "Premium 5G",
        "network_type": "5G",
        "recent_complaints": 2,
    },
    "CUST_2001": {
        "customer_id": "CUST_2001",
        "name": "Meera Shah",
        "city": "Mumbai",
        "plan": "Standard 4G",
        "network_type": "4G",
        "recent_complaints": 0,
    },
}

POLICIES = [
    {
        "city": "Pune",
        "issue_type": "slow_internet",
        "content": "If tower utilization exceeds 90 percent during peak hours, classify as likely congestion and notify NOC."
    },
    {
        "city": "Mumbai",
        "issue_type": "packet_loss",
        "content": "If packet loss exceeds 5 percent, investigate transport routes and escalate to network operations."
    },
    {
        "city": "Pune",
        "issue_type": "device_issue",
        "content": "Check APN settings, battery saver mode, and network reset status before escalation."
    },
]

ticket_counter = {"value": 5000}

## Step 6 - Failure simulation controls

In [22]:
FAILURE_FLAGS = {
    "outage_api_fail_times": 0,
    "policy_tool_fail_times": 0,
    "model_fail": False,
}

RUNTIME_COUNTERS = {
    "outage_calls": 0,
    "policy_calls": 0,
}

## Step 7 - Tool implementations

In [23]:
def get_customer_profile_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = CustomerProfileInput(**payload)
    if inp.customer_id not in CUSTOMERS:
        raise ValueError(f"Unknown customer_id: {inp.customer_id}")
    return CustomerProfileOutput(**CUSTOMERS[inp.customer_id]).model_dump()

def check_outage_primary_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = OutageCheckInput(**payload)
    RUNTIME_COUNTERS["outage_calls"] += 1

    if RUNTIME_COUNTERS["outage_calls"] <= FAILURE_FLAGS["outage_api_fail_times"]:
        raise RuntimeError("Primary outage API temporary failure")

    if inp.city == "Mumbai" and inp.issue_type == "packet_loss":
        out = OutageCheckOutput(
            city=inp.city,
            issue_type=inp.issue_type,
            outage_detected=True,
            planned_maintenance=False,
            summary="Packet loss incident active in Mumbai region."
        )
    else:
        out = OutageCheckOutput(
            city=inp.city,
            issue_type=inp.issue_type,
            outage_detected=False,
            planned_maintenance=False,
            summary="No citywide outage found."
        )
    return out.model_dump()

def check_outage_backup_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = OutageCheckInput(**payload)
    out = OutageCheckOutput(
        city=inp.city,
        issue_type=inp.issue_type,
        outage_detected=False,
        planned_maintenance=False,
        summary="Backup outage service used. No confirmed outage found."
    )
    return out.model_dump()

def search_policy_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = PolicySearchInput(**payload)
    RUNTIME_COUNTERS["policy_calls"] += 1

    if RUNTIME_COUNTERS["policy_calls"] <= FAILURE_FLAGS["policy_tool_fail_times"]:
        raise RuntimeError("Policy search tool unavailable")

    candidates = [
        p for p in POLICIES
        if (inp.city_filter is None or p["city"] == inp.city_filter)
        and (inp.issue_filter is None or p["issue_type"] == inp.issue_filter)
    ]

    if candidates:
        best = candidates[0]
        return PolicySearchOutput(top_match=best["content"], source="policy_store").model_dump()

    return PolicySearchOutput(top_match="No matching telecom policy found.", source="policy_store").model_dump()

def create_ticket_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = TicketCreateInput(**payload)
    ticket_counter["value"] += 1
    return TicketCreateOutput(
        ticket_id=f"TKT-{ticket_counter['value']}",
        status="OPEN",
        queue="NOC" if inp.priority in ("P1", "P2") else "L1_SUPPORT",
    ).model_dump()

## Step 8 - Resilience helpers

In [24]:
CIRCUIT_STATE = {
    "outage_api_failures": 0,
    "outage_api_open": False,
    "policy_failures": 0,
    "policy_open": False,
}

CIRCUIT_THRESHOLD = 2

In [25]:
def retry_tool(tool_fn, payload: Dict[str, Any], max_retries: int = 2):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return tool_fn(payload), attempt, None
        except Exception as e:
            last_error = str(e)
    return None, max_retries, last_error

def record_failure(circuit_key_failures: str, circuit_key_open: str):
    CIRCUIT_STATE[circuit_key_failures] += 1
    if CIRCUIT_STATE[circuit_key_failures] >= CIRCUIT_THRESHOLD:
        CIRCUIT_STATE[circuit_key_open] = True

def reset_circuit(circuit_key_failures: str, circuit_key_open: str):
    CIRCUIT_STATE[circuit_key_failures] = 0
    CIRCUIT_STATE[circuit_key_open] = False

def safe_model_generate(summary_text: str) -> str:
    if FAILURE_FLAGS["model_fail"]:
        raise RuntimeError("Model generation failed")
    return f"Customer-safe summary: {summary_text}"

## Step 9 - Test retry in isolation

In [26]:
FAILURE_FLAGS["outage_api_fail_times"] = 2
RUNTIME_COUNTERS["outage_calls"] = 0

result, attempts, err = retry_tool(
    check_outage_primary_tool,
    {"city": "Pune", "issue_type": "slow_internet"},
    max_retries=3
)

print("Attempts:", attempts)
print("Result:", result)
print("Error:", err)

Attempts: 3
Result: {'city': 'Pune', 'issue_type': 'slow_internet', 'outage_detected': False, 'planned_maintenance': False, 'summary': 'No citywide outage found.'}
Error: None


## Step 10 - Test fallback in isolation

In [27]:
FAILURE_FLAGS["outage_api_fail_times"] = 5
RUNTIME_COUNTERS["outage_calls"] = 0

primary_result, attempts, err = retry_tool(
    check_outage_primary_tool,
    {"city": "Pune", "issue_type": "slow_internet"},
    max_retries=2
)

if primary_result is None:
    fallback_result = check_outage_backup_tool({"city": "Pune", "issue_type": "slow_internet"})
    print("Fallback used:")
    pprint(fallback_result)
else:
    print("Primary succeeded")

Fallback used:
{'city': 'Pune',
 'issue_type': 'slow_internet',
 'outage_detected': False,
 'planned_maintenance': False,
 'summary': 'Backup outage service used. No confirmed outage found.'}


## Step 11 - LangGraph state

In [28]:
class TelecomFailureAwareState(TypedDict, total=False):
    customer_id: str
    user_query: str
    issue_type: str

    customer_profile: Dict[str, Any]
    outage_info: Dict[str, Any]
    policy_info: Dict[str, Any]

    diagnosis: str
    priority: str
    next_action: str

    ticket: Dict[str, Any]
    final_response: str

    tool_error: Optional[str]
    model_error: Optional[str]
    used_fallback: bool
    outage_attempts: int
    policy_attempts: int

## Step 12 - LangGraph nodes

In [29]:
def get_profile_node(state: TelecomFailureAwareState) -> Dict[str, Any]:
    profile = get_customer_profile_tool({"customer_id": state["customer_id"]})
    return {"customer_profile": profile}

def outage_node(state: TelecomFailureAwareState) -> Dict[str, Any]:
    city = state["customer_profile"]["city"]
    issue_type = state["issue_type"]

    if CIRCUIT_STATE["outage_api_open"]:
        backup = check_outage_backup_tool({"city": city, "issue_type": issue_type})
        return {
            "outage_info": backup,
            "used_fallback": True,
            "tool_error": "Primary outage API skipped because circuit is open.",
            "outage_attempts": 0,
        }

    result, attempts, err = retry_tool(
        check_outage_primary_tool,
        {"city": city, "issue_type": issue_type},
        max_retries=2,
    )

    if result is not None:
        reset_circuit("outage_api_failures", "outage_api_open")
        return {
            "outage_info": result,
            "used_fallback": False,
            "outage_attempts": attempts,
            "tool_error": None,
        }

    record_failure("outage_api_failures", "outage_api_open")
    backup = check_outage_backup_tool({"city": city, "issue_type": issue_type})
    return {
        "outage_info": backup,
        "used_fallback": True,
        "tool_error": f"Primary outage API failed after retries: {err}",
        "outage_attempts": attempts,
    }

def policy_node(state: TelecomFailureAwareState) -> Dict[str, Any]:
    city = state["customer_profile"]["city"]
    issue_type = state["issue_type"]

    if CIRCUIT_STATE["policy_open"]:
        return {
            "policy_info": {"top_match": "Fallback: No policy retrieved. Use standard telecom triage.", "source": "fallback"},
            "tool_error": "Policy tool skipped because circuit is open.",
            "policy_attempts": 0,
        }

    result, attempts, err = retry_tool(
        search_policy_tool,
        {
            "query_text": state["user_query"],
            "city_filter": city,
            "issue_filter": issue_type,
        },
        max_retries=2,
    )

    if result is not None:
        reset_circuit("policy_failures", "policy_open")
        return {"policy_info": result, "policy_attempts": attempts}

    record_failure("policy_failures", "policy_open")
    return {
        "policy_info": {"top_match": "Fallback: Policy unavailable, continue with outage/customer data.", "source": "fallback"},
        "tool_error": f"Policy tool failed after retries: {err}",
        "policy_attempts": attempts,
    }

def diagnose_node(state: TelecomFailureAwareState) -> Dict[str, Any]:
    profile = state["customer_profile"]
    outage = state["outage_info"]
    policy = state["policy_info"]

    if outage["outage_detected"]:
        diagnosis = f"Confirmed outage in {profile['city']}: {outage['summary']}"
        priority = "P1"
        next_action = "Escalate immediately to NOC and create a ticket."
    else:
        diagnosis = f"Likely issue guidance: {policy['top_match']}"
        priority = "P2" if profile["recent_complaints"] > 0 else "P3"
        next_action = "Use SOP guidance and monitor. Create ticket if repeated complaints continue."

    return {
        "diagnosis": diagnosis,
        "priority": priority,
        "next_action": next_action,
    }

def ticket_router(state: TelecomFailureAwareState) -> str:
    return "create_ticket" if state["priority"] in ("P1", "P2") else "generate_response"

def create_ticket_node(state: TelecomFailureAwareState) -> Dict[str, Any]:
    profile = state["customer_profile"]
    ticket = create_ticket_tool(
        {
            "customer_id": state["customer_id"],
            "city": profile["city"],
            "issue_type": state["issue_type"],
            "diagnosis": state["diagnosis"],
            "priority": state["priority"],
        }
    )
    return {"ticket": ticket}

def generate_response_node(state: TelecomFailureAwareState) -> Dict[str, Any]:
    base_summary = (
        f"Diagnosis: {state['diagnosis']} | "
        f"Priority: {state['priority']} | "
        f"Next Action: {state['next_action']}"
    )

    try:
        customer_safe = safe_model_generate(base_summary)
        response = customer_safe
        if state.get("ticket"):
            response += f" | Ticket: {state['ticket']['ticket_id']}"
        if state.get("tool_error"):
            response += f" | Note: {state['tool_error']}"
        return {"final_response": response, "model_error": None}
    except Exception as e:
        fallback = (
            f"Fallback response: {base_summary}. "
            f"Model generation failed, so a rule-based summary was returned."
        )
        if state.get("ticket"):
            fallback += f" Ticket: {state['ticket']['ticket_id']}."
        return {"final_response": fallback, "model_error": str(e)}

## Step 13 - Build the graph

In [30]:
builder = StateGraph(TelecomFailureAwareState)

builder.add_node("get_profile", get_profile_node)
builder.add_node("check_outage", outage_node)
builder.add_node("search_policy", policy_node)
builder.add_node("diagnose", diagnose_node)
builder.add_node("create_ticket", create_ticket_node)
builder.add_node("generate_response", generate_response_node)

builder.set_entry_point("get_profile")
builder.add_edge("get_profile", "check_outage")
builder.add_edge("check_outage", "search_policy")
builder.add_edge("search_policy", "diagnose")

builder.add_conditional_edges(
    "diagnose",
    ticket_router,
    {
        "create_ticket": "create_ticket",
        "generate_response": "generate_response",
    }
)

builder.add_edge("create_ticket", "generate_response")
builder.add_edge("generate_response", END)

graph = builder.compile()
print("Graph compiled")

Graph compiled


## Step 14 - Helper to reset runtime

In [31]:
def reset_runtime():
    FAILURE_FLAGS["outage_api_fail_times"] = 0
    FAILURE_FLAGS["policy_tool_fail_times"] = 0
    FAILURE_FLAGS["model_fail"] = False

    RUNTIME_COUNTERS["outage_calls"] = 0
    RUNTIME_COUNTERS["policy_calls"] = 0

    CIRCUIT_STATE["outage_api_failures"] = 0
    CIRCUIT_STATE["outage_api_open"] = False
    CIRCUIT_STATE["policy_failures"] = 0
    CIRCUIT_STATE["policy_open"] = False

reset_runtime()
print("Runtime reset")

Runtime reset


## Step 15 - Happy path run

In [32]:
reset_runtime()

state = {
    "customer_id": "CUST_1001",
    "user_query": "My mobile internet is slow in Pune. What should support do next?",
    "issue_type": "slow_internet",
}

result = graph.invoke(state)
pprint(result)

{'customer_id': 'CUST_1001',
 'customer_profile': {'city': 'Pune',
                      'customer_id': 'CUST_1001',
                      'name': 'Ravi Patil',
                      'network_type': '5G',
                      'plan': 'Premium 5G',
                      'recent_complaints': 2},
 'diagnosis': 'Likely issue guidance: If tower utilization exceeds 90 percent '
              'during peak hours, classify as likely congestion and notify '
              'NOC.',
 'final_response': 'Customer-safe summary: Diagnosis: Likely issue guidance: '
                   'If tower utilization exceeds 90 percent during peak hours, '
                   'classify as likely congestion and notify NOC. | Priority: '
                   'P2 | Next Action: Use SOP guidance and monitor. Create '
                   'ticket if repeated complaints continue. | Ticket: TKT-5001',
 'issue_type': 'slow_internet',
 'model_error': None,
 'next_action': 'Use SOP guidance and monitor. Create ticket if repeated 

## Step 16 - Simulate transient outage API failure with retry

In [33]:
reset_runtime()
FAILURE_FLAGS["outage_api_fail_times"] = 1

state = {
    "customer_id": "CUST_1001",
    "user_query": "My mobile internet is slow in Pune. What should support do next?",
    "issue_type": "slow_internet",
}

result = graph.invoke(state)
pprint(result)
print("Outage API calls:", RUNTIME_COUNTERS["outage_calls"])

{'customer_id': 'CUST_1001',
 'customer_profile': {'city': 'Pune',
                      'customer_id': 'CUST_1001',
                      'name': 'Ravi Patil',
                      'network_type': '5G',
                      'plan': 'Premium 5G',
                      'recent_complaints': 2},
 'diagnosis': 'Likely issue guidance: If tower utilization exceeds 90 percent '
              'during peak hours, classify as likely congestion and notify '
              'NOC.',
 'final_response': 'Customer-safe summary: Diagnosis: Likely issue guidance: '
                   'If tower utilization exceeds 90 percent during peak hours, '
                   'classify as likely congestion and notify NOC. | Priority: '
                   'P2 | Next Action: Use SOP guidance and monitor. Create '
                   'ticket if repeated complaints continue. | Ticket: TKT-5002',
 'issue_type': 'slow_internet',
 'model_error': None,
 'next_action': 'Use SOP guidance and monitor. Create ticket if repeated 

## Step 17 - Simulate outage API hard failure with fallback

In [34]:
reset_runtime()
FAILURE_FLAGS["outage_api_fail_times"] = 5

state = {
    "customer_id": "CUST_1001",
    "user_query": "My mobile internet is slow in Pune. What should support do next?",
    "issue_type": "slow_internet",
}

result = graph.invoke(state)
pprint(result)
print("Circuit state:", CIRCUIT_STATE)

{'customer_id': 'CUST_1001',
 'customer_profile': {'city': 'Pune',
                      'customer_id': 'CUST_1001',
                      'name': 'Ravi Patil',
                      'network_type': '5G',
                      'plan': 'Premium 5G',
                      'recent_complaints': 2},
 'diagnosis': 'Likely issue guidance: If tower utilization exceeds 90 percent '
              'during peak hours, classify as likely congestion and notify '
              'NOC.',
 'final_response': 'Customer-safe summary: Diagnosis: Likely issue guidance: '
                   'If tower utilization exceeds 90 percent during peak hours, '
                   'classify as likely congestion and notify NOC. | Priority: '
                   'P2 | Next Action: Use SOP guidance and monitor. Create '
                   'ticket if repeated complaints continue. | Ticket: TKT-5003 '
                   '| Note: Primary outage API failed after retries: Primary '
                   'outage API temporary failure

## Step 18 - Demonstrate circuit breaker behavior

In [35]:
reset_runtime()
FAILURE_FLAGS["outage_api_fail_times"] = 5

state = {
    "customer_id": "CUST_1001",
    "user_query": "My mobile internet is slow in Pune. What should support do next?",
    "issue_type": "slow_internet",
}

result1 = graph.invoke(state)
print("First run circuit:", CIRCUIT_STATE)

result2 = graph.invoke(state)
print("Second run circuit:", CIRCUIT_STATE)
pprint(result2)
print("Outage API calls made:", RUNTIME_COUNTERS["outage_calls"])

First run circuit: {'outage_api_failures': 1, 'outage_api_open': False, 'policy_failures': 0, 'policy_open': False}
Second run circuit: {'outage_api_failures': 2, 'outage_api_open': True, 'policy_failures': 0, 'policy_open': False}
{'customer_id': 'CUST_1001',
 'customer_profile': {'city': 'Pune',
                      'customer_id': 'CUST_1001',
                      'name': 'Ravi Patil',
                      'network_type': '5G',
                      'plan': 'Premium 5G',
                      'recent_complaints': 2},
 'diagnosis': 'Likely issue guidance: If tower utilization exceeds 90 percent '
              'during peak hours, classify as likely congestion and notify '
              'NOC.',
 'final_response': 'Customer-safe summary: Diagnosis: Likely issue guidance: '
                   'If tower utilization exceeds 90 percent during peak hours, '
                   'classify as likely congestion and notify NOC. | Priority: '
                   'P2 | Next Action: Use SOP guidanc

## Step 19 - Simulate policy tool failure

In [36]:
reset_runtime()
FAILURE_FLAGS["policy_tool_fail_times"] = 5

state = {
    "customer_id": "CUST_1001",
    "user_query": "My mobile internet is slow in Pune. What should support do next?",
    "issue_type": "slow_internet",
}

result = graph.invoke(state)
pprint(result)
print("Policy circuit:", CIRCUIT_STATE)

{'customer_id': 'CUST_1001',
 'customer_profile': {'city': 'Pune',
                      'customer_id': 'CUST_1001',
                      'name': 'Ravi Patil',
                      'network_type': '5G',
                      'plan': 'Premium 5G',
                      'recent_complaints': 2},
 'diagnosis': 'Likely issue guidance: Fallback: Policy unavailable, continue '
              'with outage/customer data.',
 'final_response': 'Customer-safe summary: Diagnosis: Likely issue guidance: '
                   'Fallback: Policy unavailable, continue with '
                   'outage/customer data. | Priority: P2 | Next Action: Use '
                   'SOP guidance and monitor. Create ticket if repeated '
                   'complaints continue. | Ticket: TKT-5006 | Note: Policy '
                   'tool failed after retries: Policy search tool unavailable',
 'issue_type': 'slow_internet',
 'model_error': None,
 'next_action': 'Use SOP guidance and monitor. Create ticket if repeated 

## Step 20 - Simulate model failure

In [37]:
reset_runtime()
FAILURE_FLAGS["model_fail"] = True

state = {
    "customer_id": "CUST_1001",
    "user_query": "My mobile internet is slow in Pune. What should support do next?",
    "issue_type": "slow_internet",
}

result = graph.invoke(state)
pprint(result)

{'customer_id': 'CUST_1001',
 'customer_profile': {'city': 'Pune',
                      'customer_id': 'CUST_1001',
                      'name': 'Ravi Patil',
                      'network_type': '5G',
                      'plan': 'Premium 5G',
                      'recent_complaints': 2},
 'diagnosis': 'Likely issue guidance: If tower utilization exceeds 90 percent '
              'during peak hours, classify as likely congestion and notify '
              'NOC.',
 'final_response': 'Fallback response: Diagnosis: Likely issue guidance: If '
                   'tower utilization exceeds 90 percent during peak hours, '
                   'classify as likely congestion and notify NOC. | Priority: '
                   'P2 | Next Action: Use SOP guidance and monitor. Create '
                   'ticket if repeated complaints continue.. Model generation '
                   'failed, so a rule-based summary was returned. Ticket: '
                   'TKT-5007.',
 'issue_type': 'slow_int

## Step 21 - Key learning

### Tool failure
Handled through:
- retry
- fallback
- circuit breaker

### Model failure
Handled through:
- safe generation wrapper
- rule-based final fallback response

## Step 22 - Exercises

1. Add a backup policy tool instead of a static fallback string.
2. Add a network KPI tool that can also fail.
3. Add human approval before P1 ticket creation.
4. Track failures per city.
5. Add timeout-based fallback routing.

# Summary

This notebook demonstrated:

- Retry for temporary telecom API failures
- Fallback for continuity
- Circuit breaker for repeated tool failure
- Model failure handling separate from tool failure
- Simulated API failure
- Failure-aware LangGraph workflow
- MCP-style tool catalog concepts

This is a strong foundation for reliable telecom AI agents.